In [1]:
import sys, os, re, json, shutil
from pathlib import Path
from collections import defaultdict
import csv
from urllib.parse import urlsplit, urlunsplit

# ========= 配置 =========
INPUT_DIR = Path("all_issues_new")           # issue_*.json 所在目录
OUTPUT_DIR = Path("filtered_issues_new")           # <--- MODIFIED: 修改输出目录
# 如果输出目录已存在，先删掉
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir()   # 重新创建

EXCLUDED_DIR = OUTPUT_DIR / "excluded"       # 新建一个专门存放被排除样本的子目录
EXCLUDED_DIR.mkdir()

OUT_PREFIX_SELECTED = "selected_issue_"      # 最终筛选出的样本前缀
CHUNK_SIZE = 1000

# 尝试把 stdout 固定为 UTF-8（防止终端输出报错）
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass


In [2]:
# ========= 规则 =========
# --- 必须包含的关键词 ---
RE_ACTUAL  = re.compile(r"\bactual(?:\s*(?:result|behavior|behaviour|output))?\b", re.IGNORECASE)
RE_EXPECT  = re.compile(r"\bexpect(?:ed|ation|ations|s)?(?:\s*(?:result|behavior|behaviour|output))?\b", re.IGNORECASE)
# 新：包含 Reproduction steps / How to reproduce
RE_REPRO = re.compile(
    r"\b(?:reproduction|to\s+reproduce)\b",
    re.IGNORECASE
)
EXPECTED_HEADERS = [
    r"Expected behaviour", r"Expected behavior",
    r"Expected result", r"Expected results"
]
ACTUAL_HEADERS = [
    r"Actual behaviour", r"Actual behavior",
    r"Actual result", r"Actual results"
]

RE_MARKDOWN_IMAGE = re.compile(r'!\[.*?\]\(.*?\)')
RE_HTML_IMAGE_TAG = re.compile(r'<img .*?>', re.IGNORECASE)
RE_HTML_COMMENT = re.compile(r"<!--.*?-->", re.DOTALL)
RE_HTML_TAG = re.compile(r"<[^>]+>")

# --- 必须排除的“硬”关键词 (明确是崩溃) ---
# 新：支持 crash/es/ed/ing + freeze + ANR + not responding + unresponsive + terminated
RE_CRASHY_VERB = re.compile(
    r"\b(?:crash(?:es|ed|ing)?|freeze(?:s|d|ing)?|ANR|(?:app\s+)?not\s+responding|unresponsive|terminated)\b"
    r"|[A-Za-z]+Exception\b"
    r"|\b[A-Za-z]+Error\b",
    re.IGNORECASE
)
# 同一行的崩溃例外：'no/does not/did not/never crash...' 或该行含 crash 且带 '?'
RE_NO_CRASH_LINE = re.compile(
    r"^[^\n]*\b(?:no|does\s+not|did\s+not|never)\s+crash(?:es|ed|ing)?\b[^\n]*$",
    re.IGNORECASE | re.MULTILINE
)
RE_CRASH_QMARK_LINE = re.compile(
    r"^[^\n]*\bcrash(?:es|ed|ing)?\b[^\n]*\?[^\n]*$",
    re.IGNORECASE | re.MULTILINE
)
# “Crash log” 专用
RE_CRASH_LOG_PHRASE = re.compile(r"(?<![A-Za-z0-9])crash(?:\s*|[-_])?logs?(?![A-Za-z0-9])", re.IGNORECASE)
RE_CODE_FENCE = re.compile(r"```.+?```", re.DOTALL)
RE_PLACEHOLDER_LINE = re.compile(r"(?i)^\s*(?:n/?a|no\s*response|none|N\.?A\.?)\s*$", re.MULTILINE)
RE_NEXT_HEADER = re.compile(r"(?mi)^\s*#{2,}\s")  # 下一个 Markdown 二/三级标题

def _has_nonempty_crash_log(text: str) -> bool:
    """
    发现“crash log(s)”后，截取其后的段落（到下一个 ###/## 标题或文末），
    若包含代码块 ```...``` 或典型栈迹词，视为“有真实日志”。
    """
    m = RE_CRASH_LOG_PHRASE.search(text)
    if not m:
        return False  # 没有 crash log 段
    tail = text[m.end():]
    h = RE_NEXT_HEADER.search(tail)
    seg = tail[:h.start()] if h else tail
    if not seg.strip():
        return False
    if RE_CODE_FENCE.search(seg):
        return True
    # 典型栈迹信号：Exception/Error/SIG/“ at xxx(…) ”
    for line in seg.splitlines():
        line = line.strip()
        if not line or RE_PLACEHOLDER_LINE.match(line):
            continue
        if (len(line) >= 50) or re.search(r"\b(Exception|Error|SIG[A-Z]+|stack|trace|at\s+\S+\()", line, re.IGNORECASE):
            return True
    return False

def _has_definitive_crash_line(text: str) -> bool:
    """存在至少一行明确陈述 crash（非问句，且非 'no/does not/did not/never crash...'）"""
    for line in text.splitlines():
        s = line.strip()
        if not s:
            continue
        if re.search(r'\bcrash(?:es|ed|ing)?\b', s, re.IGNORECASE):
            if '?' in s:
                continue
            if re.search(r'\b(?:no|does\s+not|did\s+not|never)\s+crash(?:es|ed|ing)?\b', s, re.IGNORECASE):
                continue
            return True
    return False

# --- 新增：剔除模板提示行（如 'Describe the bug/Crash'、带 # / ** 的标题行等）---
TEMPLATE_CRASH_PROMPT_RE = re.compile(r"""
    ^\s{0,3}                       # 开头可有少量空格
    (?:\#{1,6}\s*)?                # 可选 Markdown 标题 # ### 等
    (?:\*\*|__)?\s*                # 可选加粗起始
    (?:describe|summary)\s+        # describe / summary
    (?:the\s+)?                    # 可选 the
    (?:bug|issue|problem)          # bug / issue / problem
    (?:\s*/\s*crash)?              # 可选 '/ crash'
    (?:\s*\(.*?\))?                # 可选括号说明
    (?:\*\*|__)?\s*:?              # 可选加粗结束和冒号
    $                              # 整行
""", re.IGNORECASE | re.MULTILINE | re.VERBOSE)
LABEL_EXCLUDE_KEYWORDS = ("crash", "forceout")

# --- 软排除：改为按类别统计（不含 keyboard；keyboard 仍单独规则） ---
SOFT_EXCLUDE_CATEGORIES = {
    # 其他不符合简单复现的
    "no_in_scope": [
        "rotate", "rotation", "landscape", "portrait", "permission", "proxy", "firewall",
        "bluetooth", "NFC"
        # 注意：keyboard 不在这里，仍然走单独排除与计数
    ],
    # 规则2: 非纯文本 (视觉, 媒体等)
    "visual_media": [
        "color", "font", "size", "UI", "UX", "layout", "overlap", "look", "appearance",
        "visual", "graphic", "audio", "video", "gesture", "sound", "music",
        "picture", "thumbnail", "distracting", "blur",
        "theme", "style", "interface", "cursor", "zoom", "render"
    ],
    # 规则3: 非单设备 (需要外部环境)
    "multi_env": [
        "server", "desktop", "PC", "windows", "linux", "mac", "WebDAV",
        "nginx", "apache", "S3", "iOS", "iPad", "another device", "multiple accounts",
        "backend", "API"
    ],
    # 规则5: 非短时 (长时间等待或同步)
    "long_wait_sync": [
        "wait", "large file", "slow", "sync", "synchronization",
        "downloading", "performance", "lag", "delay"
    ],
    # 规则6: 通知栏
    "notification": [
        "notification", "status bar", "notification center"
    ],
    "display_only": ["display"],
    "ui_only": ["UI"],
    ### 不影响结果。
    # "player_only": [
    #     "player"
    # ]
    ### image 关键词相关的反而不影响
    # "image_related": [
    #     "gallery", "design", "image", 
    # ],
    ### 测试其他的。 ### 经过测试，全都不影响
    # "other": [
    #     "timeout", "uploading", "website",
    # ],
    ### pixel基本都是设备机型，不再需要排除。
    # "pixel": [
    #     "pixel"
    # ]
}

RE_FILE_SIZE = re.compile(r"\bfile\s+size\b", re.IGNORECASE)

# 预编译：为每个类别生成正则（\b 全词匹配；大小写不敏感）
def _compile_category_regex(keywords):
    # 对带空格或特殊字符的词做转义；用 \b 仅包裹词两端，允许中间有空格
    escaped = [re.escape(k) for k in keywords]
    pattern = r"\b(?:" + "|".join(escaped) + r")\b"
    return re.compile(pattern, re.IGNORECASE)

RE_SOFT_BY_CATEGORY = {cat: _compile_category_regex(kwds)
                       for cat, kwds in SOFT_EXCLUDE_CATEGORIES.items()}

# keyboard 仍保留独立逻辑与计数（规则不变）
RE_KEYBOARD = re.compile(r"keyboard", re.IGNORECASE)

RE_ANDROID_WORD = re.compile(r"\bandroid\b", re.IGNORECASE)
RE_IOS_WORD     = re.compile(r"\bios\b", re.IGNORECASE)

def _drop_lines_with_android_and_ios(text: str) -> str:
    """若一行同时含有 android 和 iOS，则整行忽略；否则保留原行。"""
    kept = []
    for line in text.splitlines():
        has_android = bool(RE_ANDROID_WORD.search(line))
        has_ios     = bool(RE_IOS_WORD.search(line))
        if has_android and has_ios:
            continue  # 忽略该行
        kept.append(line)
    return "\n".join(kept)

EXPECTED_HEADERS = [
    r"Expected behaviour", r"Expected behavior",
    r"Expected result", r"Expected results"
]
ACTUAL_HEADERS = [
    r"Actual behaviour", r"Actual behavior",
    r"Actual result", r"Actual results"
]

def _section_presence_and_nonempty(text: str, header_variants):
    """
    返回 (present, nonempty)
    - present=True 表示检测到该类标题（任一变体）
    - nonempty=True 表示该标题后的第一条“非空/非占位”行存在
    注意：如果没有匹配到任何标题（present=False），nonempty 恒为 False，但调用方会据 present 决定是否排除。
    """
    for h in header_variants:
        # 标题行（支持列表符/Markdown标题/可选冒号），独占一行
        m = re.search(rf"(?mi)^\s*(?:[-*]\s*)?(?:#+\s*)?{h}\s*:?\s*$", text)
        if not m:
            continue
        # 找到标题 → present=True
        tail = text[m.end():]
        # 在标题之后，寻找第一条“非空且非占位”的文本行
        for line in tail.splitlines():
            # 跳过空行
            if line.strip() == "":
                continue
            # 常见占位：N/A, no response, none
            if re.fullmatch(r"(?i)(?:n/?a|no\s*response|none|N\.?A\.?)", line.strip()):
                return True, False
            return True, True
        # 标题存在，但后续没有任何有效内容行
        return True, False
    # 未发现任何该类标题
    return False, False



In [3]:
def file_key(p: Path):
    m = re.search(r"issue_(\d+)\.json$", p.name, re.IGNORECASE)
    return int(m.group(1)) if m else float("inf")

def labels_have_excluded(labels):
    if not labels:
        return False
    for lbl in labels:
        name = (lbl.get("name") or "").lower()
        if any(k in name for k in LABEL_EXCLUDE_KEYWORDS):
            return True
    return False

def minimal(issue):
    return {
        "title": issue.get("title", "") or "",
        "body":  issue.get("body") or "",
        "html_url": issue.get("html_url") or ""
    }


In [4]:
# ========= 屏蔽前缀（来自 CSV）=========
CSV_PATH = Path("games_github_unique.csv")
BLOCK_PREFIXES_RAW = set()
if CSV_PATH.exists():
    with open(CSV_PATH, "r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            for cell in row:
                link = (cell or "").strip()
                if link:
                    BLOCK_PREFIXES_RAW.add(link)
else:
    print("[警告] 未找到 games_apps.csv，跳过按链接前缀排除。")

def _normalize_prefix(url: str) -> str:
    """规范化用于 startswith 比较的前缀：小写 scheme/netloc，去掉结尾斜杠"""
    try:
        parts = urlsplit(url.strip())
        scheme = parts.scheme.lower()
        netloc = parts.netloc.lower()
        path = parts.path.rstrip("/")
        return urlunsplit((scheme, netloc, path, "", ""))
    except Exception:
        return (url or "").strip().lower().rstrip("/")

def _normalize_url_for_compare(url: str) -> str:
    """规范化 issue 的 html_url：小写 scheme/netloc，去掉结尾斜杠"""
    try:
        parts = urlsplit((url or "").strip())
        return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), parts.path.rstrip("/"), "", ""))
    except Exception:
        return (url or "").strip().lower().rstrip("/")

BLOCK_PREFIXES = { _normalize_prefix(u) for u in BLOCK_PREFIXES_RAW }

def is_blocked_url(url: str) -> bool:
    """若 html_url 以任一屏蔽前缀开头（相等或 '前缀/'），则为 True"""
    nu = _normalize_url_for_compare(url)
    for p in BLOCK_PREFIXES:
        if not p:
            continue
        if nu == p or nu.startswith(p + "/"):
            return True
    return False

print(f"[信息] 已加载屏蔽前缀 {len(BLOCK_PREFIXES)} 条。")


[信息] 已加载屏蔽前缀 375 条。


In [5]:
files = sorted(INPUT_DIR.glob("issue_*.json"), key=file_key)
print(f"发现 {len(files)} 个 issue 文件，开始筛选 …")

# 每个类别维护一个列表缓冲与批次计数器
excluded_buffers = defaultdict(list)         # {category: [issue_minimal, ...]}
excluded_written_batches = defaultdict(int)  # {category: batch_count}

FLAGGED_DIR = OUTPUT_DIR / "flagged"
FLAGGED_DIR.mkdir(exist_ok=True)

flagged_buffers = defaultdict(list)          # {category: [issue_minimal, ...]}
flagged_written_batches = defaultdict(int)   # {category: batch_count}

def save_excluded_chunk(category: str, is_final_save=False):
    buf = excluded_buffers[category]
    # 如果是强制保存，或者缓冲区满了，就执行保存
    if is_final_save or (len(buf) > 0 and len(buf) % CHUNK_SIZE == 0):
        # 如果是最后保存，确定实际要保存的数量
        chunk_to_save = buf[-CHUNK_SIZE:] if not is_final_save else buf[-(len(buf) % CHUNK_SIZE) if len(buf) % CHUNK_SIZE != 0 else CHUNK_SIZE:]
        if not chunk_to_save: return # 如果没有内容可保存，则返回

        excluded_written_batches[category] += 1
        batch_num = excluded_written_batches[category]
        out_path = EXCLUDED_DIR / f"{category}-{batch_num:04d}.json"
        
        with open(out_path, "w", encoding="utf-8") as out:
            json.dump(chunk_to_save, out, ensure_ascii=False, indent=2)
        print(f"[保存-排除] {category} 第 {batch_num} 批 ({len(chunk_to_save)} 条) → {out_path}")

def save_flagged_chunk(category: str, is_final_save=False):
    buf = flagged_buffers[category]
    if is_final_save or (len(buf) > 0 and len(buf) % CHUNK_SIZE == 0):
        chunk_to_save = buf[-CHUNK_SIZE:] if not is_final_save else buf[-(len(buf) % CHUNK_SIZE) if len(buf) % CHUNK_SIZE != 0 else CHUNK_SIZE:]
        if not chunk_to_save:
            return
        flagged_written_batches[category] += 1
        batch_num = flagged_written_batches[category]
        out_path = FLAGGED_DIR / f"{category}-{batch_num:04d}.json"
        with open(out_path, "w", encoding="utf-8") as out:
            json.dump(chunk_to_save, out, ensure_ascii=False, indent=2)
        print(f"[保存-标记] {category} 第 {batch_num} 批 ({len(chunk_to_save)} 条) → {out_path}")

# --- 阶段一：初步筛选 ---
print("\n--- 阶段一：初步筛选 ---")
initial_candidates = []
for p in files:
    try:
        with open(p, "r", encoding="utf-8") as f:
            for issue in json.load(f):
                body = issue.get("body") or ""
                    # 先按 CSV 前缀排除
                html_url = issue.get("html_url") or ""
                if is_blocked_url(html_url):
                    category = "excluded_by_game_link_prefix"
                    excluded_buffers[category].append(minimal(issue))
                    save_excluded_chunk(category)
                    continue
                if RE_ACTUAL.search(body) and RE_EXPECT.search(body) and RE_REPRO.search(body):
                    # 清理模板/图片
                    cleaned_body = RE_MARKDOWN_IMAGE.sub("", body)
                    cleaned_body = RE_HTML_IMAGE_TAG.sub("", cleaned_body)
                    body_no_template = TEMPLATE_CRASH_PROMPT_RE.sub("", cleaned_body)

                    # 仅当检测到相应标题时才做“非空”校验；没有标题就不据此排除
                    exp_present, exp_nonempty = _section_presence_and_nonempty(body_no_template, EXPECTED_HEADERS)
                    act_present, act_nonempty = _section_presence_and_nonempty(body_no_template, ACTUAL_HEADERS)

                    # 规则：只在“有标题但为空”的情况下排除
                    has_empty_expected_section = exp_present and (not exp_nonempty)
                    has_empty_actual_section   = act_present and (not act_nonempty)

                    if has_empty_expected_section or has_empty_actual_section:
                        category = "excluded_by_template_sections_empty"
                        excluded_buffers[category].append(minimal(issue))
                        save_excluded_chunk(category)
                    else:
                        initial_candidates.append(issue)

    except Exception as e:
        print(f"[警告] {p.name} 读取或处理失败：{e}")
print(f"找到 {len(initial_candidates)} 条包含 'actual', 'expect', 'reproduce' 的初始候选 issue。")


# --- 阶段二：应用排除规则并统计 ---
print("\n--- 阶段二：应用排除规则 ---")
final_selected = []
exclusion_counts = defaultdict(int)

for issue in initial_candidates:
    title = issue.get("title") or ""
    body = issue.get("body") or ""
    html_url = issue.get("html_url") or ""
    
    cleaned_body = RE_MARKDOWN_IMAGE.sub("", body)
    cleaned_body = RE_HTML_IMAGE_TAG.sub("", cleaned_body) # <-- 新增这一行
    cleaned_body = RE_HTML_COMMENT.sub("", cleaned_body)      # <-- 新增：先去掉 <!-- ... -->
    cleaned_body = RE_HTML_TAG.sub(" ", cleaned_body)      # ← 新增：去掉所有剩余 HTML 标签/属性
    body_no_template = TEMPLATE_CRASH_PROMPT_RE.sub("", cleaned_body)
    
    # 规则应用
    if labels_have_excluded(issue.get("labels")):
        category = "excluded_by_label"
        exclusion_counts[category] += 1
        excluded_buffers[category].append(minimal(issue))
        save_excluded_chunk(category)
        continue
    
    # 崩溃硬排除（含 crash/es/ed/ing, freeze, ANR, not responding, unresponsive, terminated）
    combined_for_crash = (title or "") + "\n" + body_no_template
    if RE_CRASHY_VERB.search(combined_for_crash):
        # —— 新增：若出现 "crash log" 但其后没有真实日志、且除去 "crash log" 词组后不再有其它崩溃信号 → 视为例外，flag 并放行
        skip_crash_exclusion = False
        if RE_CRASH_LOG_PHRASE.search(body_no_template):
            # 去掉 "crash log" 词组再看是否还有其它崩溃信号（避免把有 ANR/freeze 等真崩溃的误放）
            text_wo_crashlog = RE_CRASH_LOG_PHRASE.sub("", body_no_template)
            combined_wo_crashlog = (title or "") + "\n" + text_wo_crashlog
            has_other_crash_tokens = bool(RE_CRASHY_VERB.search(combined_wo_crashlog))

            if not _has_nonempty_crash_log(body_no_template) and not has_other_crash_tokens:
                category = "crash_log_empty"
                flagged_buffers[category].append(minimal(issue))
                save_flagged_chunk(category)
                skip_crash_exclusion = True  # 当作例外：不按崩溃排除

        if not skip_crash_exclusion:
            # 既有“真实 crash log”，或没有 crash log 段，走原有的否定/问号例外 -> 否则排除
            if (RE_NO_CRASH_LINE.search(body_no_template) or RE_CRASH_QMARK_LINE.search(body_no_template)) and not _has_definitive_crash_line(body_no_template):
                category = "crash_neg_or_question"
                flagged_buffers[category].append(minimal(issue))
                save_flagged_chunk(category)
                # 不 continue：后续还能走 keyboard/soft
            else:
                category = "excluded_by_crashy_verb"
                exclusion_counts[category] += 1
                excluded_buffers[category].append(minimal(issue))
                save_excluded_chunk(category)
                continue
        
    if RE_KEYBOARD.search(title) or RE_KEYBOARD.search(cleaned_body) or RE_KEYBOARD.search(html_url):
        category = "excluded_by_keyboard"
        exclusion_counts[category] += 1
        excluded_buffers[category].append(minimal(issue))
        save_excluded_chunk(category)
        continue

    soft_hit_cat = None
    soft_hit_word = None

    for cat, regex in RE_SOFT_BY_CATEGORY.items():
        # 对 multi_env：忽略同一行同时含 Android 和 iOS 的那几行
        if cat == "multi_env":
            title_for_cat = _drop_lines_with_android_and_ios(title or "")
            body_for_cat  = _drop_lines_with_android_and_ios(cleaned_body)
        else:
            title_for_cat = title or ""
            body_for_cat  = cleaned_body

        m = regex.search(title_for_cat)
        if not m:
            m = regex.search(body_for_cat)

        if m:
            soft_hit_cat = cat
            soft_hit_word = (m.group(0) or "").lower()
            break
        
    if soft_hit_cat:
        # 例外1：visual_media 命中 'size'，且全文出现 'file size' → flag 放行
        if soft_hit_cat == "visual_media" and soft_hit_word == "size" and RE_FILE_SIZE.search(title + "\n" + cleaned_body):
            category = "visual_media_size_filecontext"
            flagged_buffers[category].append(minimal(issue))
            save_flagged_chunk(category)
            # 不 continue，放行到最终选中
            
        elif soft_hit_cat == "display_only":
            category = "display_only_flag"
            flagged_buffers[category].append(minimal(issue))
            save_flagged_chunk(category)
            # 不 continue：走到循环末尾，由兜底 append 进 final_selected

        elif soft_hit_cat == "ui_only":
            category = "ui_only_flag"
            flagged_buffers[category].append(minimal(issue))
            save_flagged_chunk(category)
            # 不 continue：走到循环末尾，由兜底 append 进 final_selected

        # 其他软排除：照旧排除
        else:
            exclusion_counts[soft_hit_cat] += 1
            excluded_buffers[soft_hit_cat].append(minimal(issue))
            save_excluded_chunk(soft_hit_cat)
            continue
        
    # 这里补上兜底，把没被 continue 的样本加入最终结果
    final_selected.append(minimal(issue))

发现 519 个 issue 文件，开始筛选 …

--- 阶段一：初步筛选 ---
[保存-排除] excluded_by_game_link_prefix 第 1 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0001.json
[保存-排除] excluded_by_game_link_prefix 第 2 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0002.json
[保存-排除] excluded_by_game_link_prefix 第 3 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0003.json
[保存-排除] excluded_by_game_link_prefix 第 4 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0004.json
[保存-排除] excluded_by_game_link_prefix 第 5 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0005.json
[保存-排除] excluded_by_game_link_prefix 第 6 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0006.json
[保存-排除] excluded_by_game_link_prefix 第 7 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0007.json
[保存-排除] excluded_by_game_link_prefix 第 8 批 (1000 条) → filtered_issues_new/excluded/excluded_by_game_lin

In [6]:

# --- 阶段三：报告结果并保存 ---
print("\n--- 阶段三：结果报告 ---")
print(f"初始候选集数量: {len(initial_candidates)}")
print("排除统计:")
for reason, count in sorted(exclusion_counts.items()):
    print(f"  - {reason}: {count}")
total_excluded = sum(exclusion_counts.values())
print(f"总共排除数量: {total_excluded}")
print(f"最终筛选出数量: {len(final_selected)}")



--- 阶段三：结果报告 ---
初始候选集数量: 30262
排除统计:
  - excluded_by_crashy_verb: 8506
  - excluded_by_keyboard: 1090
  - excluded_by_label: 62
  - long_wait_sync: 555
  - multi_env: 3927
  - no_in_scope: 1894
  - notification: 261
  - visual_media: 7442
总共排除数量: 23737
最终筛选出数量: 6525


In [7]:

# --- 保存最终筛选出的文件 ---
print("\n--- 开始保存最终筛选出的文件 ---")
written_batches = 0
for i in range(0, len(final_selected), CHUNK_SIZE):
    chunk = final_selected[i:i + CHUNK_SIZE]
    written_batches += 1
    out_path = OUTPUT_DIR / f"{OUT_PREFIX_SELECTED}{written_batches:04d}.json"
    with open(out_path, "w", encoding="utf-8") as out:
        json.dump(chunk, out, ensure_ascii=False, indent=2)
    print(f"[保存-选中] 第 {written_batches} 批 ({len(chunk)} 条) → {out_path}")

# --- 保存所有被排除的剩余文件 ---
print("\n--- 开始保存剩余的被排除文件 ---")
for category in excluded_buffers:
    save_excluded_chunk(category, is_final_save=True)
# --- 保存所有已标记(flagged)的剩余文件 ---
print("\n--- 开始保存剩余的专门标记文件 ---")
for category in flagged_buffers:
    save_flagged_chunk(category, is_final_save=True)

# 可选：输出 flagged 统计，方便抽样核对
print("\n标记统计(仅供抽样复核)：")
for cat, buf in flagged_buffers.items():
    print(f"  - {cat}: {len(buf)}")
    
print("\n筛选完成！")



--- 开始保存最终筛选出的文件 ---
[保存-选中] 第 1 批 (1000 条) → filtered_issues_new/selected_issue_0001.json
[保存-选中] 第 2 批 (1000 条) → filtered_issues_new/selected_issue_0002.json
[保存-选中] 第 3 批 (1000 条) → filtered_issues_new/selected_issue_0003.json
[保存-选中] 第 4 批 (1000 条) → filtered_issues_new/selected_issue_0004.json
[保存-选中] 第 5 批 (1000 条) → filtered_issues_new/selected_issue_0005.json
[保存-选中] 第 6 批 (1000 条) → filtered_issues_new/selected_issue_0006.json
[保存-选中] 第 7 批 (525 条) → filtered_issues_new/selected_issue_0007.json

--- 开始保存剩余的被排除文件 ---
[保存-排除] excluded_by_game_link_prefix 第 59 批 (431 条) → filtered_issues_new/excluded/excluded_by_game_link_prefix-0059.json
[保存-排除] excluded_by_template_sections_empty 第 1 批 (54 条) → filtered_issues_new/excluded/excluded_by_template_sections_empty-0001.json
[保存-排除] no_in_scope 第 2 批 (894 条) → filtered_issues_new/excluded/no_in_scope-0002.json
[保存-排除] excluded_by_label 第 1 批 (62 条) → filtered_issues_new/excluded/excluded_by_label-0001.json
[保存-排除] excluded_by_crashy